# 📊 PCA reducción de dimensionalidad paso a paso

Aprenderemos PCA desde cero usando **datos simulados**.

## Objetivos
- Entender qué problema resuelve PCA.
- Comprender por qué estandarizamos.
- Entender qué son PC1, PC2 y los *loadings*.
- Interpretar la varianza explicada.
- Determinar cuántas componentes conservar.
- Reducir 10 variables a 2 dimensiones y visualizar los datos.

> ⚠️ Los datos de este notebook son simulados y no representan un conjunto de datos real.

# 1. Antes de PCA: ¿qué significa tener muchas variables?

Imaginemos que queremos describir a un grupo de estudiantes.

Tenemos 10 variables: horas de estudio, tareas, asistencia, notas y hábitos.

Muchas pueden estar relacionadas. Por ejemplo, quien estudia más puede tender a tener mejores calificaciones.

**Pregunta:** ¿podemos representar gran parte de esa información usando menos dimensiones?

# 2. Importamos las librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")

print("Librerías cargadas correctamente.")

: 

# 3. Generamos datos simulados

Crearemos 300 estudiantes.

Para que las variables estén relacionadas, construiremos primero dos **factores latentes**:

- `factor_academico`
- `factor_habitos`

Luego utilizaremos estos factores para generar 10 variables observables.

La idea es simular que detrás de muchas variables observadas existen algunos patrones comunes.

In [ ]:
n_students = 300

factor_academico = np.random.normal(
    loc=0, scale=1, size=n_students
)

factor_habitos = np.random.normal(
    loc=0, scale=1, size=n_students
)

print("Factor académico:", factor_academico[:10])
print("Factor hábitos:", factor_habitos[:10])

## ¿Qué acabamos de crear?

Podemos imaginar:

```text
                 Factor académico
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
       Tareas      Calificaciones  Asistencia


                  Factor hábitos
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
       Sueño       Participación  Organización
```

Ahora construiremos las 10 variables observables a partir de esos factores.

In [ ]:
noise = np.random.normal(
    loc=0,
    scale=0.4,
    size=(n_students, 10)
)

X = np.column_stack([
    0.90 * factor_academico + 0.20 * factor_habitos,
    0.80 * factor_academico + 0.30 * factor_habitos,
    0.70 * factor_academico + 0.40 * factor_habitos,
    0.60 * factor_academico + 0.50 * factor_habitos,
    0.50 * factor_academico + 0.60 * factor_habitos,
    0.40 * factor_academico + 0.70 * factor_habitos,
    0.30 * factor_academico + 0.80 * factor_habitos,
    0.20 * factor_academico + 0.90 * factor_habitos,
    0.70 * factor_academico - 0.20 * factor_habitos,
    0.20 * factor_academico + 0.70 * factor_habitos
]) + noise

feature_names = [
    "Horas_estudio", "Tareas", "Asistencia", "Matematicas",
    "Programacion", "Fisica", "Estadistica", "Participacion",
    "Rendimiento", "Organizacion"
]

df = pd.DataFrame(X, columns=feature_names)
df.head()

# 4. Conocemos nuestros datos

Cada fila representa un estudiante y cada columna una variable.

In [ ]:
print("Número de estudiantes:", df.shape[0])
print("Número de variables:", df.shape[1])
print("Dimensiones:", df.shape)

display(df.describe().round(2))

# 5. ¿Las variables están relacionadas?

La correlación nos permite detectar relaciones lineales entre variables.

- Cerca de **+1** → relación positiva fuerte.
- Cerca de **0** → poca relación lineal.
- Cerca de **−1** → relación negativa fuerte.

Si existen muchas correlaciones, puede haber información redundante.

In [ ]:
plt.figure(figsize=(12, 9))

correlation_matrix = df.corr()

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Matriz de correlación entre las variables")
plt.tight_layout()
plt.show()

# 6. Visualizamos dos variables

Veamos una relación sencilla: horas de estudio vs. matemáticas.

Si existe un patrón, veremos que los puntos tienden a seguir alguna dirección.

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    df["Horas_estudio"],
    df["Matematicas"],
    alpha=0.6
)

plt.xlabel("Horas de estudio")
plt.ylabel("Matemáticas")
plt.title("Relación entre dos variables")
plt.show()

# 7. La intuición de PCA

Observa la nube de puntos.

PCA busca **direcciones de máxima variación**.

La primera dirección se llama:

# PC1 — Principal Component 1

PC1 es la dirección que captura la mayor cantidad posible de variabilidad de los datos.

Después encuentra una segunda dirección, PC2, perpendicular a la primera, que captura la mayor variabilidad restante, y así sucesivamente.

# 8. ¿Por qué debemos estandarizar?

Las variables pueden estar en escalas diferentes.

Por ejemplo:

```text
Edad       → 18 a 70
Ingresos   → 1,000 a 100,000
```

Una variable con números grandes podría dominar el análisis.

Por eso normalmente estandarizamos para que cada variable tenga aproximadamente:

- media = 0
- desviación estándar = 1

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(df)

display(
    pd.DataFrame(
        X_scaled,
        columns=feature_names
    ).head().round(2)
)

In [ ]:
scaled_df = pd.DataFrame(X_scaled, columns=feature_names)

print("Medias:")
print(scaled_df.mean().round(3))

print("\nDesviaciones estándar:")
print(scaled_df.std().round(3))

# 9. Ahora sí: aplicamos PCA

Primero utilizaremos PCA sin fijar el número de componentes.

Tenemos 10 variables originales, por lo que podemos obtener hasta 10 componentes:

```text
Datos originales
      ↓
Estandarización
      ↓
      PCA
      ↓
PC1, PC2, PC3, ..., PC10
```

Las componentes quedan ordenadas de acuerdo con la cantidad de variabilidad que explican.

In [ ]:
pca = PCA()

X_pca = pca.fit_transform(X_scaled)

print("Dimensiones originales:", X_scaled.shape)
print("Dimensiones después de PCA:", X_pca.shape)

# 10. ¿Cuánta varianza explica cada componente?

La **varianza explicada** indica qué proporción de la variabilidad total representa cada componente.

Si PC1 explica 60 %, significa que esa primera dirección conserva el 60 % de la variabilidad observada en los datos.

In [ ]:
explained_variance = pca.explained_variance_ratio_

for i, variance in enumerate(explained_variance):
    print(
        f"PC{i+1}: {variance:.4f} "
        f"({variance*100:.2f}%)"
    )

In [ ]:
components = np.arange(1, len(explained_variance) + 1)

plt.figure(figsize=(9, 5))

plt.bar(
    components,
    explained_variance * 100
)

plt.xlabel("Componente principal")
plt.ylabel("Varianza explicada (%)")
plt.title("Varianza explicada por cada componente")
plt.xticks(components)
plt.show()

# 11. Varianza explicada acumulada

Ahora queremos responder:

> ¿Cuántas componentes necesito para conservar, por ejemplo, el 95 % de la variabilidad?

Para ello calculamos la suma acumulada de la varianza explicada.

In [ ]:
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

for i, variance in enumerate(cumulative_variance):
    print(
        f"PC1 hasta PC{i+1}: "
        f"{variance*100:.2f}%"
    )

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    components,
    cumulative_variance * 100,
    marker="o"
)

plt.axhline(
    y=95,
    linestyle="--",
    label="95%"
)

plt.xlabel("Número de componentes")
plt.ylabel("Varianza acumulada (%)")
plt.title("Varianza explicada acumulada")
plt.xticks(components)
plt.ylim(0, 105)
plt.legend()
plt.show()

In [ ]:
n_components_95 = np.argmax(
    cumulative_variance >= 0.95
) + 1

print(
    "Número de componentes necesarias para explicar "
    f"al menos el 95%: {n_components_95}"
)

# 12. ¿Qué son realmente PC1 y PC2?

PC1 y PC2 **no son variables originales**.

Son combinaciones de las variables originales.

Por ejemplo:

$$
PC1 = 0.40X_1 + 0.38X_2 + 0.35X_3 + \cdots
$$

Los coeficientes se llaman **cargas o loadings**.

Cuanto mayor sea el valor absoluto de una carga, mayor es la contribución de esa variable a la componente.

> El signo puede cambiar por la orientación matemática de la componente; lo importante es analizar magnitudes y patrones.

In [ ]:
components_matrix = pd.DataFrame(
    pca.components_,
    columns=feature_names,
    index=[f"PC{i+1}" for i in range(len(pca.components_))]
)

display(components_matrix.round(3))

In [ ]:
plt.figure(figsize=(12, 6))

sns.heatmap(
    components_matrix.iloc[:4],
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Loadings de las primeras 4 componentes")
plt.xlabel("Variables originales")
plt.ylabel("Componentes principales")
plt.tight_layout()
plt.show()

# 13. Reducimos de 10 variables a 2 componentes

Ahora queremos visualizar los datos.

No podemos visualizar directamente 10 dimensiones, pero sí podemos representar los datos en 2 dimensiones:

```text
10 variables
      ↓
     PCA
      ↓
PC1 + PC2
      ↓
  gráfico 2D
```

In [ ]:
pca_2d = PCA(n_components=2)

X_2d = pca_2d.fit_transform(X_scaled)

print("Dimensiones originales:", X_scaled.shape)
print("Dimensiones reducidas:", X_2d.shape)

In [ ]:
df_pca = pd.DataFrame(
    X_2d,
    columns=["PC1", "PC2"]
)

display(df_pca.head())

In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(
    df_pca["PC1"],
    df_pca["PC2"],
    alpha=0.65
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Datos proyectados sobre PC1 y PC2")

plt.axhline(0, linestyle="--", alpha=0.5)
plt.axvline(0, linestyle="--", alpha=0.5)

plt.show()

# 14. ¿Cuánta información conservamos usando solo PC1 y PC2?

No basta decir que pasamos de 10 variables a 2.

Debemos preguntar:

> ¿Cuánta varianza conservamos?

In [ ]:
variance_2d = pca_2d.explained_variance_ratio_

print(f"PC1 explica: {variance_2d[0]*100:.2f}%")
print(f"PC2 explica: {variance_2d[1]*100:.2f}%")
print(f"PC1 + PC2 explican: {variance_2d.sum()*100:.2f}%")

# 15. ¿PCA elimina variables?

**No exactamente.**

PCA no selecciona simplemente algunas variables.

### Selección de variables

```text
X1 X2 X3 X4 X5
 ↓     ↓
X1    X3
```

### PCA

```text
X1 ──┐
X2 ──┤
X3 ──┼──→ PC1
X4 ──┤
X5 ──┘

y también crea PC2, PC3, ...
```

PCA crea **nuevas variables** a partir de combinaciones de las originales.

# 16. ¿Cuándo puede ser útil PCA?

PCA puede ser útil cuando:

- Tenemos muchas variables.
- Muchas variables están correlacionadas.
- Queremos visualizar datos de alta dimensión.
- Queremos reducir redundancia.
- Queremos utilizar las componentes como entrada de un modelo.

Pero PCA no siempre mejora un modelo de Machine Learning. Depende del problema y del objetivo.

# 17. ¿Qué perdemos al reducir dimensiones?

Si pasamos de:

```text
10 variables → 2 componentes
```

normalmente perdemos cierta información.

La pregunta correcta es:

> ¿Cuánta información perdimos y es aceptable?

Por ejemplo, si PC1 + PC2 conservan 92 %:

```text
Información conservada     → 92 %
Información no conservada  →  8 %
```

# 18. PCA explicado con una analogía

Imagina un objeto 3D.

Si tomamos una fotografía desde un mal ángulo, perdemos mucha información.

PCA busca los ángulos o direcciones desde los cuales podemos representar la mayor cantidad posible de información.

Por eso podemos pensar en PCA como una **proyección inteligente de los datos**.

# 19. Resumen visual del proceso

```text
                 DATOS
                   │
                   ▼
          10 variables originales
                   │
                   ▼
             StandardScaler
                   │
                   ▼
             Datos escalados
                   │
                   ▼
                  PCA
                   │
          ┌────────┼────────┐
          ▼        ▼        ▼
         PC1      PC2      PC3 ...
          │        │
          └────┬───┘
               ▼
          2 dimensiones
               │
               ▼
          Visualización
```

# 20. PCA completo en pocas líneas

Una vez entendida la teoría, el flujo básico es:

```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
```

Y para saber cuánta variabilidad conservamos:

```python
pca.explained_variance_ratio_
```

In [ ]:
# Pipeline completo

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_

print("Varianza explicada por PC1:",
      round(explained[0] * 100, 2), "%")

print("Varianza explicada por PC2:",
      round(explained[1] * 100, 2), "%")

print("Varianza explicada total:",
      round(explained.sum() * 100, 2), "%")

In [ ]:
plt.figure(figsize=(10, 7))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    alpha=0.7
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA: representación de 10 variables en 2 dimensiones")

plt.axhline(0, linestyle="--", alpha=0.5)
plt.axvline(0, linestyle="--", alpha=0.5)

plt.show()

# 🧠 21. PCA en 5 minutos

### 1️⃣ Tenemos muchas variables

```text
X1 X2 X3 X4 X5 X6 X7 X8 X9 X10
```

### 2️⃣ Algunas contienen información relacionada

Puede existir redundancia.

### 3️⃣ Estandarizamos

```python
X_scaled = StandardScaler().fit_transform(X)
```

### 4️⃣ PCA encuentra nuevas direcciones

```python
pca = PCA()
```

Estas direcciones son:

```text
PC1
PC2
PC3
...
```

### 5️⃣ PC1 explica la mayor variabilidad

PC2 explica la siguiente mayor cantidad, etc.

### 6️⃣ Podemos quedarnos con pocas componentes

```text
10 variables → 2 componentes
```

### 7️⃣ Comprobamos cuánto conservamos

```python
pca.explained_variance_ratio_
```

## ⭐ Idea principal

> **PCA transforma muchas variables posiblemente correlacionadas en un conjunto menor de nuevas variables llamadas componentes principales, intentando conservar la mayor cantidad posible de variabilidad.**

# 📝 22. Actividad para el estudiante

Responde utilizando los resultados obtenidos:

1. ¿Cuántas variables tenía originalmente el dataset?
2. ¿Por qué estandarizamos los datos antes de aplicar PCA?
3. ¿Qué porcentaje de varianza explica PC1?
4. ¿Qué porcentaje explican conjuntamente PC1 y PC2?
5. ¿Cuántas componentes necesitamos para explicar al menos el 95 %?
6. ¿PC1 corresponde a una variable original? Explica.
7. ¿PCA elimina variables? Diferencia PCA de selección de variables.
8. ¿Qué ventajas tiene representar los datos con 2 componentes?
9. ¿Qué desventaja tiene reducir de 10 variables a 2?
10. Si PC1 explica 70 % y PC2 20 %, ¿qué porcentaje explica PC1 + PC2?

# 🎯 23. Reto final

## Experimento 1

Cambia:

```python
PCA(n_components=2)
```

por:

```python
PCA(n_components=3)
```

¿Qué cambia?

## Experimento 2

Utiliza:

```python
PCA(n_components=0.95)
```

¿Qué ocurre?

## Experimento 3

Compara:

```text
10 variables originales
```

contra:

```text
2 componentes
```

¿Cuánto se redujo la dimensionalidad?

## Experimento 4

Elimina la estandarización y vuelve a ejecutar PCA.

¿Cambian los resultados? ¿Por qué?

In [ ]:
# PCA automático: conservar al menos el 95% de la varianza

pca_95 = PCA(n_components=0.95)

X_pca_95 = pca_95.fit_transform(X_scaled)

print("Número de componentes seleccionadas:",
      pca_95.n_components_)

print("Varianza conservada:",
      round(pca_95.explained_variance_ratio_.sum() * 100, 2),
      "%")

print("Dimensiones finales:",
      X_pca_95.shape)

# 🏁 24. Conclusión

PCA cambia la forma en que representamos nuestros datos.

Partimos de:

**10 variables**

y encontramos nuevas variables:

**PC1, PC2, PC3, ...**

ordenadas según la cantidad de variabilidad que explican.

## ⭐ Qué debes recordar

**PCA no selecciona simplemente algunas variables.**

**PCA crea nuevas variables a partir de las originales.**

Estas nuevas variables intentan conservar la mayor cantidad posible de información sobre la variabilidad de los datos.

### En una frase

> **PCA es una técnica de reducción de dimensionalidad que transforma variables originales en componentes principales, ordenadas según la cantidad de varianza que explican.**